# XRD Rietveld Plot Generator

Publication-quality Rietveld plots from the **CSV that the GSAS-II Rietveld
plot saves** - batch processing, built-in validation, cross-platform.

Not the file from *Export → Powder data as → histogram CSV file*: that one
has a quoted preamble and different column names, and is rejected.

Full documentation (input format, usage, configuration, privacy notes):
see [`README.md`](README.md).

## 1. Setup

Dependency check, then the engine. Parsing, data preparation, plotting and
the batch driver live in [`xrd_plotter.py`](xrd_plotter.py), imported here
as `xp`. Plot appearance (2θ window, colours, line widths, fonts) is set by
the constants at the top of that file and overridden on the module, as the
cell below shows. Input format and numerical-precision details are in the
README.

In [ ]:
# Dependency bootstrap - installs only what is missing. IPython arrives with
# any Jupyter kernel, and is listed so an editor resolves it as well.
import importlib.util, subprocess, sys

for module, package in (("numpy", "numpy"), ("pandas", "pandas"),
                        ("matplotlib", "matplotlib"),
                        ("ipywidgets", "ipywidgets"), ("IPython", "ipython"),
                        ("pytest", "pytest")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "--quiet", package])
print("Dependencies OK")

In [ ]:
"""The plotting engine lives in xrd_plotter.py; this cell loads it."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xrd_plotter as xp

# Appearance is set by the constants in the module. Override them here, on
# the module itself, so every routine sees the change:
#   xp.PLOT_X_MIN, xp.PLOT_X_MAX = 13, 85   # fix the 2theta window
#   xp.WEIGHTED_RESIDUALS = False           # raw diff in the lower panel
#   xp.PHASE_COLORS = {"phase 1": "#1f77b4"}
print("Engine loaded:", Path(xp.__file__).name)


## 2. Validation (self-test on synthetic data)

Runs [`test_xrd_plotter.py`](test_xrd_plotter.py). The suite rebuilds
synthetic GSAS-II-style exports and asserts bit-exact parsing in both
separator and decimal-mark variants, detection of the phase columns among a
full set of export columns, isolation of corrupt, ragged and incomplete
files, the order and colours of the phases, the metadata binding into the
legend, the 2theta window, the unweighted residual panel, the interactive
helper, and a batch that survives one unusable file.

The cell fails on the first failing assertion, so running the notebook is a
test run, and so is `pytest -q` from a terminal. Only synthetic data is
used.

In [ ]:
import pytest

# The suite builds its own synthetic exports, so this cell reads nothing
# from data/ and works on a fresh clone.
exit_code = pytest.main(["-q", "--no-header", "test_xrd_plotter.py"])
assert exit_code == 0, "the validation suite failed, see the report above"
print("\nALL VALIDATION CHECKS PASSED")

## 3. Plot your own exports

Copy your CSV exports into `data/`, optionally place `Samples_metadata.csv`
next to the notebook, adjust the four settings below and run. Each figure
appears here as it is drawn, with the files it was written to named
underneath, so a long folder reports itself while it runs. Everything
lands in `output/` as a PDF and a 600 dpi PNG. Step-by-step instructions
are in the README.

> **Keep your data private:** `data/`, `output/` and `Samples_metadata.csv`
> are listed in `.gitignore` and must never be committed or uploaded.

In [ ]:
DATA_FOLDER = "data"                       # your GSAS-II CSV exports
METADATA_FILE = "Samples_metadata.csv"     # optional, PRIVATE - never commit
OUTPUT_FOLDER = "output"                   # created automatically
USE_SQRT = True                            # False -> linear intensity axis

results = xp.process_folder(DATA_FOLDER, METADATA_FILE, OUTPUT_FOLDER,
                         use_sqrt=USE_SQRT)

## 4. Try a different window on one file

Pick a file and type the limits. The figure redraws as soon as a control
changes: a box on Enter or when you leave it, a checkbox and the dropdown
at once. **Apply** redraws on demand. Nothing is saved: this is the place
to find the window you want before running the batch again.

An empty box leaves that end of the axis to the setting behind it, the
`xp.PLOT_X_MIN` and `xp.PLOT_X_MAX` constants for 2θ and the data itself
for the intensity.

The panel prints a metadata line under the figure. Paste it into
`Samples_metadata.csv` and section 3 will draw that sample this way every
time.

This section needs `ipywidgets`, which the first cell installs along with
the other dependencies. Without it the section prints how to install it
and the rest of the notebook is unaffected.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run 'pip install ipywidgets', "
          "then re-run this cell.")

files = sorted(f for f in Path(DATA_FOLDER).glob("*.csv")
               if f.name != Path(METADATA_FILE).name)

if widgets is None or not files:
    if widgets is not None:
        print(f"No CSV files in '{DATA_FOLDER}': nothing to replot.")
else:
    picker = widgets.Dropdown(options=[(f.name, str(f)) for f in files],
                              description="File:")
    # continuous_update=False: a box redraws when you press Enter or leave
    # it, not on every keystroke.
    boxes = {k: widgets.Text(description=d, placeholder="auto",
                             continuous_update=False,
                             layout=widgets.Layout(width="180px"))
             for k, d in (("x_min", "2theta min"), ("x_max", "2theta max"),
                          ("y_min", "y min"), ("y_max", "y max"))}
    sqrt_box = widgets.Checkbox(value=USE_SQRT, description="sqrt intensity")
    weighted_box = widgets.Checkbox(value=xp.WEIGHTED_RESIDUALS,
                                    description="diff/sigma")
    apply_button = widgets.Button(description="Apply", button_style="primary")
    out = widgets.Output()

    def redraw(_=None):
        """Draw the picked file with whatever the controls now hold."""
        out.clear_output(wait=True)  # replace the previous figure in place
        with out:
            limits = {k: xp.to_number(b.value) if b.value.strip() else None
                      for k, b in boxes.items()}
            try:
                fig, line = xp.replot_file(picker.value, METADATA_FILE,
                                           use_sqrt=sqrt_box.value,
                                           weighted=weighted_box.value,
                                           **limits)
            except ValueError as e:
                print(f"Cannot draw this file: {e}")
                return
            plt.show()
            plt.close(fig)
            print("Metadata line for this window:\n" + line)

    # Every control redraws on its own, and Apply stays for a second look.
    for control in (picker, sqrt_box, weighted_box, *boxes.values()):
        control.observe(redraw, names="value")
    apply_button.on_click(redraw)

    display(widgets.VBox([
        picker,
        widgets.HBox([boxes["x_min"], boxes["x_max"]]),
        widgets.HBox([boxes["y_min"], boxes["y_max"]]),
        widgets.HBox([sqrt_box, weighted_box, apply_button]),
        out,
    ]))
    redraw()  # start on the first file instead of an empty panel